# 🏓 TTNet – Player Performance Analysis
**نظام تحليل أداء لاعب تنس الطاولة**

---
### خطوات هذا الـ Notebook
1. ✅ ربط Google Drive
2. ✅ رفع الكود
3. ✅ تثبيت المتطلبات
4. ✅ تحميل الداتا
5. ✅ استخراج الفريمات
6. ✅ التدريب (3 مراحل)
7. ✅ تشغيل تحليل أداء اللاعب

> **ملاحظة:** تأكدي من تفعيل GPU قبل البدء:
> `Runtime → Change runtime type → T4 GPU`

## 0️⃣ تحقق من الـ GPU

In [ ]:
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')
else:
    print('⚠️  لا يوجد GPU! تأكدي من Runtime → Change runtime type → T4 GPU')

## ⚙️ إعدادات المشروع — شغّلي هذه الخلية أولاً دائماً
> إذا انقطعت الجلسة أو أعدتِ تشغيل الـ Kernel، شغّلي هذه الخلية مباشرةً قبل أي خلية أخرى.

In [ ]:
import os
from pathlib import Path

# ══════════════════════════════════════════════════════════════════
#   ← غيري القيم هنا فقط إذا أردتِ
# ══════════════════════════════════════════════════════════════════

PROJECT_DIR  = '/content/drive/MyDrive/TTNet'   # مجلد المشروع على Drive
TRAIN_GAME   = 'game_1'                          # game_1 إلى game_5
TEST_GAME    = 'test_2'                          # test_2 = أصغر حجم (0.2 GB)

# ══════════════════════════════════════════════════════════════════
#   متغيرات مشتقة تلقائياً — لا تعدّليها
# ══════════════════════════════════════════════════════════════════

DATASET_DIR     = f'{PROJECT_DIR}/dataset'
CODE_DIR        = f'{PROJECT_DIR}/code'
CHECKPOINTS_DIR = f'{PROJECT_DIR}/checkpoints'
LOGS_DIR        = f'{PROJECT_DIR}/logs'
RESULTS_DIR     = f'{PROJECT_DIR}/results'
BASE_URL        = 'https://lab.osai.ai/datasets/openttgames/data/'

# SRC_DIR يُحسب بعد رفع الكود — هنا نحاول اكتشافه تلقائياً
def _find_src(base):
    for root, dirs, files in os.walk(base):
        if 'main.py' in files and 'config' in dirs:
            return root
    return None

SRC_DIR  = _find_src(CODE_DIR) if os.path.isdir(CODE_DIR) else None
ROOT_DIR = str(Path(SRC_DIR).parent) if SRC_DIR else CODE_DIR

# إنشاء المجلدات الأساسية
for d in [PROJECT_DIR, DATASET_DIR, CODE_DIR,
          CHECKPOINTS_DIR, LOGS_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ── ملخص ──────────────────────────────────────────────────────────
print('✅ الإعدادات جاهزة:')
print(f'   PROJECT_DIR  = {PROJECT_DIR}')
print(f'   DATASET_DIR  = {DATASET_DIR}')
print(f'   TRAIN_GAME   = {TRAIN_GAME}')
print(f'   TEST_GAME    = {TEST_GAME}')
print(f'   SRC_DIR      = {SRC_DIR or "لم يُعثر عليه بعد (ارفعي الكود أولاً)"}')

## 1️⃣ ربط Google Drive
كل الملفات (كود، داتا، نماذج) ستُحفظ على Drive بحيث لا تضيع عند إغلاق الجلسة.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive مربوط. مجلد المشروع:', PROJECT_DIR)

## 2️⃣ رفع الكود

**عندك خيارين:**

### الخيار أ – رفع ملف ZIP (الأسرع)
1. اضغطي على مجلد المشروع على جهازك
2. اعمليه ZIP
3. ارفعيه على: `MyDrive/TTNet/TTNet.zip`
4. شغّلي الخلية أدناه

### الخيار ب – GitHub (لو رفعتي الكود)
```python
!git clone https://github.com/YOUR_USERNAME/TTNet.git {PROJECT_DIR}/code
```

In [ ]:
# ── الخيار أ: فك ضغط الـ ZIP ──────────────────────────────────────────────────
ZIP_PATH  = f'{PROJECT_DIR}/TTNet.zip'
CODE_DIR  = f'{PROJECT_DIR}/code'

if os.path.isfile(ZIP_PATH):
    print('📦 فك ضغط الكود...')
    !unzip -q "{ZIP_PATH}" -d "{CODE_DIR}"
    print('✅ تم فك الضغط في:', CODE_DIR)
elif os.path.isdir(CODE_DIR):
    print('✅ الكود موجود مسبقاً في:', CODE_DIR)
else:
    print('⚠️  لم يُعثر على ZIP. ارفعي الملف على Drive أو استخدمي GitHub.')

# ── الخيار ب: GitHub ───────────────────────────────────────────────────────────
# GITHUB_URL = 'https://github.com/YOUR_USERNAME/TTNet.git'
# !git clone {GITHUB_URL} {CODE_DIR}

## 3️⃣ تثبيت المتطلبات

In [ ]:
print('📦 تثبيت المكتبات المطلوبة...')

# المكتبات الأساسية (Colab لديه torch مثبت مسبقاً)
!pip install -q easydict==1.9 scikit-learn tqdm

# TurboJPEG – ضروري لقراءة الصور بسرعة
!apt-get install -q -y libturbojpeg
!pip install -q PyTurboJPEG

# tensorboard
!pip install -q tensorboard

print('✅ جميع المكتبات مثبتة')

## 4️⃣ إصلاح مشاكل التوافق مع NumPy الحديث
الكود الأصلي يستخدم `np.int` المحذوف من NumPy 1.24+. نصلح هذا تلقائياً.

In [ ]:
import subprocess, os

# اجد المسار الصحيح للكود
# الكود قد يكون في مجلد فرعي داخل code/ حسب اسم ZIP
def find_src_dir(base):
    for root, dirs, files in os.walk(base):
        if 'main.py' in files and 'config' in dirs:
            return root
    return None

SRC_DIR = find_src_dir(CODE_DIR)
if SRC_DIR is None:
    # fallback: try common patterns
    for sub in os.listdir(CODE_DIR):
        candidate = os.path.join(CODE_DIR, sub, 'src')
        if os.path.isdir(candidate):
            SRC_DIR = candidate
            break

print('📁 SRC_DIR:', SRC_DIR)
ROOT_DIR = os.path.dirname(SRC_DIR) if SRC_DIR else CODE_DIR

# إصلاح np.int → int في جميع ملفات Python
import re
from pathlib import Path

fixed = []
for py_file in Path(CODE_DIR).rglob('*.py'):
    text = py_file.read_text(errors='ignore')
    new_text = re.sub(r'dtype=np\.int\b', 'dtype=np.int32', text)
    new_text = re.sub(r'\.astype\(np\.int\)', '.astype(np.int32)', new_text)
    new_text = re.sub(r'np\.int\b(?!\d|e|_)', 'int', new_text)
    if new_text != text:
        py_file.write_text(new_text)
        fixed.append(str(py_file))

if fixed:
    print(f'✅ تم إصلاح {len(fixed)} ملفات:')
    for f in fixed:
        print('   ', f)
else:
    print('✅ لا توجد مشاكل توافق')

## 5️⃣ تحميل الداتا (OpenTTGames) – فيديو واحد فقط

المصدر: [lab.osai.ai](https://lab.osai.ai/)

اختاري **فيديو تدريب واحد** و**فيديو اختبار واحد** من الجدول التالي:

| الاسم | الحجم | النوع |
|-------|-------|-------|
| `game_1` | 5.6 GB | تدريب |
| `game_2` | 10.8 GB | تدريب |
| `game_3` | 4.6 GB | تدريب |
| `game_4` | 3.9 GB | تدريب |
| `game_5` | 4.5 GB | تدريب |
| `test_2` | **0.2 GB** ✅ أصغر اختبار | اختبار |
| `test_3` | 0.5 GB | اختبار |
| `test_1` | 1.1 GB | اختبار |

> **نصيحة:** استخدمي `game_1` للتدريب و`test_2` للاختبار – هما الأنسب للكولاب المجاني.

In [ ]:
# المجلدات
for d in ['training/videos', 'training/annotations',
          'test/videos',     'test/annotations']:
    os.makedirs(f'{DATASET_DIR}/{d}', exist_ok=True)

def download_if_missing(url, dst, label):
    if os.path.isfile(dst):
        size_mb = os.path.getsize(dst) / 1e6
        print(f'  [skip] {label} موجود مسبقاً ({size_mb:.0f} MB)')
        return
    print(f'  ⬇️  تحميل {label} ...')
    import subprocess
    subprocess.run(['wget', '-q', '--show-progress', url, '-O', dst], check=True)
    size_mb = os.path.getsize(dst) / 1e6
    print(f'  ✅ تم تحميل {label} ({size_mb:.0f} MB)')

# ── فيديو التدريب ─────────────────────────────────────────────────────────────
print(f'📥 فيديو التدريب: {TRAIN_GAME}')
download_if_missing(
    url = f'{BASE_URL}{TRAIN_GAME}.mp4',
    dst = f'{DATASET_DIR}/training/videos/{TRAIN_GAME}.mp4',
    label = f'{TRAIN_GAME}.mp4'
)
download_if_missing(
    url = f'{BASE_URL}{TRAIN_GAME}.zip',
    dst = f'{DATASET_DIR}/training/annotations/{TRAIN_GAME}.zip',
    label = f'{TRAIN_GAME}.zip (annotations)'
)

# ── فيديو الاختبار ────────────────────────────────────────────────────────────
print(f'📥 فيديو الاختبار: {TEST_GAME}')
download_if_missing(
    url = f'{BASE_URL}{TEST_GAME}.mp4',
    dst = f'{DATASET_DIR}/test/videos/{TEST_GAME}.mp4',
    label = f'{TEST_GAME}.mp4'
)
download_if_missing(
    url = f'{BASE_URL}{TEST_GAME}.zip',
    dst = f'{DATASET_DIR}/test/annotations/{TEST_GAME}.zip',
    label = f'{TEST_GAME}.zip (annotations)'
)

print(f'\n✅ اكتمل التحميل')
print(f'   تدريب: {TRAIN_GAME} | اختبار: {TEST_GAME}')

## 6️⃣ فك ضغط الـ Annotations

In [ ]:
import zipfile

def unzip_annotations(annos_dir):
    for zip_fn in os.listdir(annos_dir):
        if not zip_fn.endswith('.zip'):
            continue
        zip_path   = os.path.join(annos_dir, zip_fn)
        out_folder = os.path.join(annos_dir, zip_fn[:-4])
        if os.path.isdir(out_folder):
            print(f'  [skip] {zip_fn} مفكوك مسبقاً')
            continue
        print(f'  📂 فك {zip_fn} ...')
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(annos_dir)

print('📂 فك ضغط annotations التدريب...')
unzip_annotations(f'{DATASET_DIR}/training/annotations')
print('📂 فك ضغط annotations الاختبار...')
unzip_annotations(f'{DATASET_DIR}/test/annotations')
print('✅ تم فك الضغط')

## 6️⃣ب ضبط الـ Config على الفيديو الواحد المحمّل
الكود الأصلي يتوقع 5 فيديوهات تدريب + 7 اختبار. هنا نضبطه تلقائياً على الفيديو اللي حملناه.

In [ ]:
from pathlib import Path
import re

# مسار config.py
config_path = Path(SRC_DIR) / 'config' / 'config.py'
config_text = config_path.read_text()

# استبدال قوائم الألعاب المُشفّرة بالفيديو الواحد المحمّل
config_text = re.sub(
    r"configs\.train_game_list\s*=\s*\[.*?\]",
    f"configs.train_game_list = ['{TRAIN_GAME}']",
    config_text
)
config_text = re.sub(
    r"configs\.test_game_list\s*=\s*\[.*?\]",
    f"configs.test_game_list = ['{TEST_GAME}']",
    config_text
)

config_path.write_text(config_text)

# تحقق
for line in config_text.splitlines():
    if 'game_list' in line:
        print(' ', line.strip())

print('\n✅ config.py تم ضبطه على:')
print(f'   train_game_list = [{TRAIN_GAME}]')
print(f'   test_game_list  = [{TEST_GAME}]')

## 7️⃣ استخراج الفريمات
نستخرج فقط الفريمات المطلوبة بحد أقصى **128 فريم** لكل فيديو مع التركيز على **اللاعب الأيسر**.

In [ ]:
import sys, glob as _glob
PREPARE_DIR = os.path.join(ROOT_DIR, 'prepare_dataset')
sys.path.insert(0, PREPARE_DIR)

%cd "{PREPARE_DIR}"

# استخراج فيديو التدريب فقط
print(f'🖼️  استخراج فريمات {TRAIN_GAME} (تدريب) ...')
!python extract_selected_images.py \
    --dataset_dir "{DATASET_DIR}" \
    --dataset_types training \
    --max_frames 128 \
    --left_player \
    --num_frames_sequence 9

# استخراج فيديو الاختبار فقط
print(f'🖼️  استخراج فريمات {TEST_GAME} (اختبار) ...')
!python extract_selected_images.py \
    --dataset_dir "{DATASET_DIR}" \
    --dataset_types test \
    --max_frames 128 \
    --left_player \
    --num_frames_sequence 9

# تحقق من عدد الصور المستخرجة
train_imgs = _glob.glob(f'{DATASET_DIR}/training/images/{TRAIN_GAME}/*.jpg')
test_imgs  = _glob.glob(f'{DATASET_DIR}/test/images/{TEST_GAME}/*.jpg')
print(f'\n✅ اكتمل استخراج الفريمات:')
print(f'   {TRAIN_GAME} (تدريب) : {len(train_imgs)} صورة')
print(f'   {TEST_GAME}  (اختبار): {len(test_imgs)} صورة')

## 8️⃣ إعداد مجلدات الإخراج

In [ ]:
print('📁 المجلدات:')
print('   Checkpoints:', CHECKPOINTS_DIR)
print('   Logs:       ', LOGS_DIR)
print('   Results:    ', RESULTS_DIR)

## 9️⃣ التدريب – المرحلة الأولى
**Global Ball Detection + Segmentation فقط**

- `--no_local` `--no_event` → لا تدريب على Local أو Events
- `--global_weight 2` → تركيز على الكرة العالمية
- `--num_epochs 30`

In [ ]:
%cd "{SRC_DIR}"

!python main.py \
    --working-dir "{PROJECT_DIR}" \
    --saved_fn ttnet_phase1 \
    --gpu_idx 0 \
    --batch_size 8 \
    --num_epochs 30 \
    --lr 1e-3 \
    --lr_type plateau \
    --no_local \
    --no_event \
    --global_weight 2 \
    --seg_weight 1 \
    --smooth-labelling \
    --earlystop_patience 5 \
    --checkpoint_freq 2 \
    --num_workers 2

print('✅ انتهى تدريب المرحلة الأولى')

## 🔟 التدريب – المرحلة الثانية
**Local Ball Detection + Event Spotting**

- نحمّل نتائج المرحلة الأولى (`pretrained_path`)
- `--overwrite_global_2_local` → ننسخ أوزان Global لـ Local
- نجمّد Global + Segmentation

In [ ]:
PHASE1_CKPT = f'{CHECKPOINTS_DIR}/ttnet_phase1/ttnet_phase1_best.pth'

if not os.path.isfile(PHASE1_CKPT):
    # ابحثي عن أحدث checkpoint
    ckpt_dir = f'{CHECKPOINTS_DIR}/ttnet_phase1'
    ckpts    = sorted([
        f for f in os.listdir(ckpt_dir) if f.endswith('.pth')
    ]) if os.path.isdir(ckpt_dir) else []
    PHASE1_CKPT = os.path.join(ckpt_dir, ckpts[-1]) if ckpts else ''

print('📌 Checkpoint المرحلة الأولى:', PHASE1_CKPT)

%cd "{SRC_DIR}"

!python main.py \
    --working-dir "{PROJECT_DIR}" \
    --saved_fn ttnet_phase2 \
    --gpu_idx 0 \
    --batch_size 8 \
    --num_epochs 30 \
    --lr 1e-3 \
    --lr_type plateau \
    --pretrained_path "{PHASE1_CKPT}" \
    --overwrite_global_2_local \
    --freeze_global \
    --freeze_seg \
    --local_weight 1 \
    --event_weight 1 \
    --smooth-labelling \
    --earlystop_patience 5 \
    --num_workers 2

print('✅ انتهى تدريب المرحلة الثانية')

## 1️⃣1️⃣ التدريب – المرحلة الثالثة (Fine-tuning الكل)
نفتح جميع الأوزان للتدريب معاً بـ learning rate منخفض.

In [ ]:
PHASE2_CKPT = f'{CHECKPOINTS_DIR}/ttnet_phase2/ttnet_phase2_best.pth'

if not os.path.isfile(PHASE2_CKPT):
    ckpt_dir = f'{CHECKPOINTS_DIR}/ttnet_phase2'
    ckpts    = sorted([
        f for f in os.listdir(ckpt_dir) if f.endswith('.pth')
    ]) if os.path.isdir(ckpt_dir) else []
    PHASE2_CKPT = os.path.join(ckpt_dir, ckpts[-1]) if ckpts else ''

print('📌 Checkpoint المرحلة الثانية:', PHASE2_CKPT)

%cd "{SRC_DIR}"

!python main.py \
    --working-dir "{PROJECT_DIR}" \
    --saved_fn ttnet_phase3 \
    --gpu_idx 0 \
    --batch_size 8 \
    --num_epochs 30 \
    --lr 1e-4 \
    --lr_type plateau \
    --pretrained_path "{PHASE2_CKPT}" \
    --global_weight 1 \
    --local_weight 1 \
    --event_weight 1 \
    --seg_weight 1 \
    --smooth-labelling \
    --earlystop_patience 7 \
    --num_workers 2

print('✅ انتهى التدريب الكامل')

## 1️⃣2️⃣ مراقبة التدريب بـ TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{LOGS_DIR}"

## 1️⃣3️⃣ اختبار الموديل (Test)

In [ ]:
PHASE3_CKPT = f'{CHECKPOINTS_DIR}/ttnet_phase3/ttnet_phase3_best.pth'

if not os.path.isfile(PHASE3_CKPT):
    ckpt_dir = f'{CHECKPOINTS_DIR}/ttnet_phase3'
    ckpts    = sorted([
        f for f in os.listdir(ckpt_dir) if f.endswith('.pth')
    ]) if os.path.isdir(ckpt_dir) else []
    PHASE3_CKPT = os.path.join(ckpt_dir, ckpts[-1]) if ckpts else ''

print('📌 Best Checkpoint:', PHASE3_CKPT)

%cd "{SRC_DIR}"

!python test.py \
    --working-dir "{PROJECT_DIR}" \
    --gpu_idx 0 \
    --pretrained_path "{PHASE3_CKPT}" \
    --save_test_output

## 1️⃣4️⃣ تشغيل تحليل أداء اللاعب

**رفعي فيديو اللاعب على Drive** ثم حددي المسار أدناه.

- `--assessment_video` : فيديو التقييم الأولي (10 كرات)
- `--stage_videos`     : فيديوهات المراحل
- `--starting_level`  : مستوى اللاعب (beginner/intermediate/advanced)

In [ ]:
# ── حدّدي مسارات الفيديوهات ───────────────────────────────────────────────────
PLAYER_VIDEO = f'{PROJECT_DIR}/videos/player_game.mp4'   # ← غيري هذا

# للتحقق من وجود الفيديو
if not os.path.isfile(PLAYER_VIDEO):
    print('⚠️  الفيديو غير موجود على المسار:', PLAYER_VIDEO)
    print('ارفعيه على Drive أو غيري المسار أعلاه')
else:
    print('✅ الفيديو موجود')

# ── تحميل الفيديو مباشرة من الجهاز (اختياري) ─────────────────────────────────
# from google.colab import files
# uploaded = files.upload()
# PLAYER_VIDEO = list(uploaded.keys())[0]

In [ ]:
%cd "{SRC_DIR}"

# ── تشغيل نظام تحليل الأداء ──────────────────────────────────────────────────
!python performance_demo.py \
    --pretrained_path "{PHASE3_CKPT}" \
    --assessment_video "{PLAYER_VIDEO}" \
    --stage_videos "{PLAYER_VIDEO}" \
    --gpu_idx 0 \
    --max_retries 3 \
    --bounce_thresh 0.5 \
    --net_thresh 0.5

## 1️⃣5️⃣ تشغيل مباشر (Demo أصلي)
لمشاهدة تتبع الكرة والـ Segmentation على الفيديو.

In [ ]:
%cd "{SRC_DIR}"

DEMO_OUTPUT = f'{RESULTS_DIR}/demo'
os.makedirs(DEMO_OUTPUT, exist_ok=True)

!python demo.py \
    --working-dir "{PROJECT_DIR}" \
    --pretrained_path "{PHASE3_CKPT}" \
    --video_path "{PLAYER_VIDEO}" \
    --gpu_idx 0 \
    --output_format video \
    --save_demo_output

print('✅ Demo اكتمل. الفيديو محفوظ في:', DEMO_OUTPUT)

## 1️⃣6️⃣ تحميل النتائج
لتحميل الـ checkpoint وفيديو النتائج على جهازك.

In [ ]:
from google.colab import files

# ── تحميل Best Checkpoint ──────────────────────────────────────────────────────
if os.path.isfile(PHASE3_CKPT):
    print('⬇️  تحميل Best Checkpoint...')
    files.download(PHASE3_CKPT)

# ── تحميل فيديو Demo ──────────────────────────────────────────────────────────
demo_video = os.path.join(DEMO_OUTPUT, 'ttnet_phase3', 'result.mp4')
if os.path.isfile(demo_video):
    print('⬇️  تحميل فيديو Demo...')
    files.download(demo_video)
else:
    print('ℹ️  فيديو Demo غير موجود (ربما التدريب لم يكتمل بعد)')

---
## 📋 ملاحظات مهمة

| الموضوع | التفاصيل |
|---------|----------|
| **وقت التدريب** | ~2-4 ساعات لكل مرحلة على T4 GPU |
| **حفظ الجلسة** | كل شيء محفوظ على Drive – لو انقطعت الجلسة تقدري تكملي من آخر checkpoint |
| **استكمال التدريب** | استخدمي `--resume_path` بدل `--pretrained_path` |
| **Colab Pro** | يعطيك وقت جلسة أطول وـ A100 GPU أسرع |
| **الداتا** | ~10 GB كاملة، أو تقدري تشتغلي بـ game_1 فقط للاختبار |

### للاستكمال من checkpoint:
```python
# بدّلي --pretrained_path بـ --resume_path
!python main.py \
    --resume_path "{CHECKPOINTS_DIR}/ttnet_phase1/ttnet_phase1.pth" \
    ...
```